In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score


In [ ]:
data_path = "../../data/Student_performance_data _.csv"
df = pd.read_csv(data_path)
print(df.shape)
print(df.columns.tolist())


In [ ]:
target_col = "GradeClass"
true_labels = df[target_col].values
X_cluster = df.drop(columns=["StudentID", target_col])
print("Features for clustering:", X_cluster.columns.tolist())
print("X_cluster shape:", X_cluster.shape)


In [ ]:
scaler = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
X_scaled = scaler.fit_transform(X_cluster)


In [ ]:
n_clusters_list = [3, 4, 5, 6]
inertias = []
silhouettes = []
aris = []
nmis = []

for k in n_clusters_list:
    model = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42)
    labels = model.fit_predict(X_scaled)
    inertias.append(model.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))
    aris.append(adjusted_rand_score(true_labels, labels))
    nmis.append(normalized_mutual_info_score(true_labels, labels))
    print(f"k={k} | silhouette={silhouettes[-1]:.4f} | ARI={aris[-1]:.4f} | NMI={nmis[-1]:.4f}")


In [ ]:
best_k = n_clusters_list[int(np.argmax(silhouettes))]
print("Best k by silhouette:", best_k)
kmeans_final = KMeans(n_clusters=best_k, init="k-means++", n_init=10, random_state=42)
cluster_labels = kmeans_final.fit_predict(X_scaled)


In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

os.makedirs("visualizations", exist_ok=True)
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap="tab10", alpha=0.6, s=20)
plt.colorbar(scatter, label="Cluster")
plt.title(f"K-Means Clustering (k={best_k}) - PCA 2D")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.savefig("visualizations/kmeans_clusters_pca.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
results = pd.DataFrame({
    "Algorithm": ["KMeans"],
    "Best_k": [best_k],
    "Silhouette": [silhouette_score(X_scaled, cluster_labels)],
    "ARI": [adjusted_rand_score(true_labels, cluster_labels)],
    "NMI": [normalized_mutual_info_score(true_labels, cluster_labels)]
})
os.makedirs("results", exist_ok=True)
results.to_csv("results/kmeans_results.csv", index=False)
print(results)
